# Part 7 — Retrieval Deep Dive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-07-retrieval-deep-dive/rag_hybrid.ipynb)

*Dense and sparse retrieval fail in opposite directions — so combine them.*

📖 Read the essay: https://www.mefby.com/essays/retrieval-deep-dive  
🐍 Companion script: `rag_hybrid.py`

In Part 6 we built a working RAG app, but it retrieves naively: pure semantic (dense) search with a fixed top-k. That quietly fails on a whole class of queries — the ones that hinge on an exact code, name, or ID. This notebook fixes that by hand.

**What you'll build**

- A tiny support knowledge base where the chunk that answers the query is *terse and code-heavy* — exactly what dense search blurs.
- A **dense** retriever (cosine over embeddings) and watch it bury the right answer.
- A compact **BM25** sparse retriever from scratch (TF-IDF refined with saturation + length normalization) and watch it nail the exact code.
- Two ways to **fuse** them into hybrid search: a normalized **weighted sum** and **Reciprocal Rank Fusion (RRF)**.
- The **top-k** knob, demonstrated as a real trade-off (recall vs. noise).

**Runs fully offline.** Only `numpy` is required. If a real embedding model is installed and cached it is used automatically; otherwise a transparent, deterministic stand-in keeps every cell executable with no network and no API key.

## Setup

Everything we need from the standard library (`math`, `re`, `collections`) plus `numpy`. No other dependencies — the whole point of this part is that sparse retrieval and fusion are pure, transparent arithmetic.

In [ ]:
import math
import re
from collections import Counter

import numpy as np

print('setup OK — stdlib + numpy', np.__version__)

## The miss that motivates everything

Here is the same little support knowledge base our Part 6 app indexed. Read **chunk 0** closely: it literally answers the query, but it is terse and code-heavy. That terseness is its undoing under dense search — a dense model *rounds off* a rare alphanumeric token like `E-4042`, blurring it into the gist of the sentence.

A user types the most natural question in the world. You'd bet money the app returns chunk 0. We'll see that it does not.

In [ ]:
# The same support knowledge base from Part 6. Chunk 0 is the answer.
CORPUS = [
    "Error E-4042: the authentication token has expired. Refresh the token and retry the request.",
    "Troubleshooting checkout and payment failures: common causes and first steps.",
    "Resolving login and authentication issues for returning customers.",
    "The checkout page shows a generic error after the customer clicks Pay.",
    "Contact our support team about an existing order or delivery.",
    "Refund policy: refunds are accepted within 30 days of purchase.",
]

QUERY = "how do I fix error E-4042 at checkout?"

for i, c in enumerate(CORPUS):
    print(f'{i}. {c}')
print()
print('QUERY:', repr(QUERY))

## Dense (semantic) retrieval: recap and limits

**Dense retrieval** is the approach from Parts 2 and 3: embed the query and every chunk into vectors, then rank chunks by cosine similarity to the query. It is *dense* because the vectors are short and packed — every dimension carries a real number, and meaning is spread across all of them at once.

Its strength is meaning: it matches *automobile* to *car* for free. Its weakness is the exact flip side. Meaning becomes geometry, and geometry is *smooth* — built to treat near-synonyms as nearly identical. That same smoothing blurs an exact, rare token (`E-4042`, an SKU, a name) into the gist of the sentence, so it effectively vanishes from the match.

In your real app, the dense scores come straight from Part 6's `dense_search` (cosine over `sentence-transformers` embeddings). We keep that as the **headline** path below, wrapped in a guard so this notebook still runs with **no model and no network**: if the real embedder can't load, we drop to the deterministic demo scores the `.py` ships with. Those demo numbers mimic what a dense model returns for this query — on-topic chunks score high, the exact-code chunk gets blurred down to rank 5.

In [ ]:
# The intended path (what you'd ship): import dense_search from Part 6, which
# does cosine over real sentence-transformers embeddings.
#
#     from rag import dense_search           # Part 6's cosine-over-embeddings
#     dense = dense_search(QUERY, CORPUS)
#
# We wrap the load in try/except so the demo is runnable offline. If the real
# model / Part-6 module is unavailable, we fall back to the deterministic demo
# scores below (same numbers the .py uses).

# Deterministic stand-in scores: what a dense model roughly returns for QUERY.
# Note chunk 0 (the E-4042 answer) is blurred DOWN to 0.55 -> it lands at rank 5.
_DEMO_DENSE = [0.55, 0.88, 0.80, 0.70, 0.62, 0.45]


def load_dense_search():
    """Return a real (query, corpus) -> scores function, or None if offline."""
    try:
        from sentence_transformers import SentenceTransformer  # the real path
        model = SentenceTransformer('all-MiniLM-L6-v2')        # 384-dim vectors

        def dense_search(query, corpus):
            q = model.encode([query], normalize_embeddings=True)[0]
            docs = model.encode(list(corpus), normalize_embeddings=True)
            return [float(np.dot(q, d)) for d in docs]  # cosine (unit vectors)

        return dense_search
    except Exception as exc:  # no package, no network, no cached weights...
        print(f'[real model unavailable: {type(exc).__name__}] -> offline fallback')
        return None


_real = load_dense_search()
if _real is not None:
    dense_search = _real
    print('Dense embedder in use: REAL all-MiniLM-L6-v2')
else:
    def dense_search(query, corpus):
        return list(_DEMO_DENSE)
    print('Dense embedder in use: offline deterministic stand-in')

Let's score every chunk with dense retrieval and look at its top 3. We'll use two small helpers — `order` (doc indices sorted best-first) and `show` (print the top 3 of a ranking) — for the rest of the notebook.

Watch where chunk 0 lands. To keep the lesson reproducible regardless of which embedder is active, the printed ranking uses the deterministic demo scores; the real model lands in the same neighborhood (the exact code is rounded off either way).

In [ ]:
def order(scores):
    """Doc indices sorted by score, best first."""
    return sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)


def show(title, ranked, scores):
    print(title)
    for rank, i in enumerate(ranked[:3], start=1):
        print(f'  {rank}. [{scores[i]:.3f}] {CORPUS[i][:54]}')
    print()


# We compute real dense scores when a model is present, but pin the printed
# ranking to the deterministic demo scores so the teaching numbers are stable.
dense_real = dense_search(QUERY, CORPUS)   # used by the real path in your app
dense = list(_DEMO_DENSE)                  # deterministic for this lesson

dense_rank = order(dense)
show('DENSE only (meaning):', dense_rank, dense)
print('Where did the E-4042 chunk (index 0) land? rank',
      dense_rank.index(0) + 1, '-> below a top-3 cutoff, so the generator never sees it.')

## Sparse (keyword / lexical) retrieval

**Sparse retrieval** (also called keyword or lexical retrieval) scores a chunk by how much its actual *words* overlap with the query's actual words. No embeddings, no geometry of meaning — just terms matching terms. It is the mirror image of dense: imagine one slot per distinct vocabulary word (tens of thousands), of which a chunk uses only a few dozen, so the vector is almost all zeros — hence *sparse*.

First, tokenization. The one subtlety that matters here: we keep a hyphenated code like `e-4042` intact as a **single token** rather than splitting it on the hyphen. That rare token is the whole point — we must not shred it.

In [ ]:
def tokenize(text):
    # keep hyphenated codes like 'e-4042' intact as one token
    return re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', text.lower())


print('chunk 0 tokens :', tokenize(CORPUS[0]))
print('query   tokens :', tokenize(QUERY))
print()
print("'e-4042' survives as one token:", 'e-4042' in tokenize(CORPUS[0]))

### From TF-IDF to BM25

The simplest sparse weighting is **TF-IDF**, the product of two intuitive quantities:

- **Term frequency (TF)** — how often a term appears *in this chunk*. More occurrences, higher weight.
- **Inverse document frequency (IDF)** — how *rare* a term is across the whole collection. Common words like *the* are crushed toward zero; a rare token like `e-4042` is enormously informative and gets a high weight.

Almost nobody ships raw TF-IDF anymore — they ship **BM25**, its robust refinement and the default ranking function in Lucene, Elasticsearch, and friends. We build it in two steps. First the constructor: tokenize the corpus, measure document lengths, and precompute **IDF** from each term's document frequency (how many chunks contain it).

In [ ]:
class BM25:
    """Okapi BM25. k1 controls term-frequency saturation;
    b controls how much we normalize for document length."""

    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [tokenize(d) for d in corpus]
        self.N = len(self.docs)
        self.doc_len = [len(d) for d in self.docs]
        self.avgdl = sum(self.doc_len) / self.N
        # document frequency: how many docs contain each term
        df = Counter()
        for d in self.docs:
            for term in set(d):
                df[term] += 1
        # inverse document frequency: rare terms (small df) weigh far more
        self.idf = {
            term: math.log(1 + (self.N - n + 0.5) / (n + 0.5))
            for term, n in df.items()
        }

    def scores(self, query):
        q = tokenize(query)
        out = [0.0] * self.N
        for i, doc in enumerate(self.docs):
            tf = Counter(doc)
            for term in q:
                if term not in tf:
                    continue
                freq = tf[term]
                # saturation: more occurrences help less and less (k1)
                numer = freq * (self.k1 + 1)
                # length normalization: long docs don't get an unfair edge (b)
                denom = freq + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl)
                out[i] += self.idf.get(term, 0.0) * numer / denom
        return out


bm25 = BM25(CORPUS)
# Show how decisive the rare code is: high IDF vs. a common word.
print(f"IDF('e-4042')  = {bm25.idf.get('e-4042', 0.0):.3f}   (rare -> high weight)")
print(f"IDF('the')     = {bm25.idf.get('the', 0.0):.3f}   (common -> low weight)")
print(f"IDF('checkout')= {bm25.idf.get('checkout', 0.0):.3f}")

The `scores` method above bakes in the two corrections that make BM25 dependable. You don't need the formula memorized, just the two intuitions:

- **Term-frequency saturation (`k1`).** Plain TF grows without limit, which is silly — mentioning *refund* twenty times shouldn't score ten times higher than twice. The `k1` term makes each extra occurrence matter less and less: diminishing returns, the way a human reader judges relevance.
- **Document-length normalization (`b`).** A long chunk contains more words, so by sheer length it's likelier to contain your query terms by accident. The `b` term normalizes for length so a long chunk gets no unfair edge over a short, focused one.

BM25 is excellent at exactly what dense is worst at: exact matching of keywords, codes, names. Its weakness is the inverse — it is literal, so it can't connect *car* to *automobile* (**vocabulary mismatch**).

Now score the same query with BM25 and look at its top 3. The chunk dense buried should rocket to the top — the rare token `e-4042` is decisive.

In [ ]:
sparse = bm25.scores(QUERY)
sparse_rank = order(sparse)
show('SPARSE only (BM25 keywords):', sparse_rank, sparse)
print('Where did the E-4042 chunk (index 0) land now? rank',
      sparse_rank.index(0) + 1, '<- found it.')

## The complementarity insight

Put the two failure modes side by side and the chapter clicks into place:

- **Dense** understands meaning but is fuzzy on exact terms.
- **Sparse** matches exact terms but is blind to meaning.

They don't fail in similar ways — they fail in *opposite directions*. Where one is weak, the other is precisely strong. When two methods fail in opposite directions, the move is obvious: stop choosing between them and **combine** them.

## Hybrid search

**Hybrid search** runs both retrievers over the same query and merges their results into a single ranked list. The only real question is *how* to merge. There are two common strategies, and we'll build both.

### Strategy 1: weighted sum (and why you must normalize first)

The direct approach: combine each retriever's scores into one number per chunk and re-rank, using a convex combination governed by a knob called **alpha**:

```
combined = alpha * dense_score + (1 - alpha) * sparse_score
```

There's one catch you cannot skip: **dense and sparse scores live on completely different scales.** Cosine sits near 0..1; BM25 is unbounded and runs into the single or double digits (look at the sparse scores above — `2.253` vs. dense's `0.880`). Add them raw and BM25 simply drowns out cosine; alpha becomes meaningless. So first we **normalize** each list onto a common 0..1 range with min-max scaling.

In [ ]:
def min_max(scores):
    """Squash a score list into 0..1 so dense and sparse become comparable.
    Mandatory before any weighted sum: the two live on different scales."""
    lo, hi = min(scores), max(scores)
    if hi == lo:
        return [0.0 for _ in scores]
    return [(s - lo) / (hi - lo) for s in scores]


print('dense  raw :', [round(x, 3) for x in dense])
print('dense  0..1:', [round(x, 3) for x in min_max(dense)])
print('sparse raw :', [round(x, 3) for x in sparse])
print('sparse 0..1:', [round(x, 3) for x in min_max(sparse)])

With both lists on the same scale, the weighted sum is honest. `alpha=1` is pure dense, `alpha=0` is pure sparse, `0.5` splits the difference.

In [ ]:
def weighted_fusion(dense, sparse, alpha=0.5):
    """Convex combination AFTER normalization. alpha=1 -> pure dense,
    alpha=0 -> pure sparse."""
    d, s = min_max(dense), min_max(sparse)   # MUST normalize before summing
    return [alpha * d[i] + (1 - alpha) * s[i] for i in range(len(d))]


w = weighted_fusion(dense, sparse, alpha=0.5)
show('HYBRID via weighted sum (alpha=0.5):', order(w), w)
print('E-4042 chunk (index 0) is now at rank', order(w).index(0) + 1, '-> rescued into the top 3.')

### Strategy 2: Reciprocal Rank Fusion (RRF)

The second strategy sidesteps the scale problem entirely by throwing away the raw scores and merging by **rank** instead. **Reciprocal Rank Fusion** gives each chunk points based on its *position* in each list: rank 1 earns `1 / (k + 1)`, rank 2 earns `1 / (k + 2)`, and so on, where `k` is a small constant (60 is the common default). Sum a chunk's points across both lists and re-rank by the total.

The intuition: appearing near the top of *either* list is worth a lot, and near the top of *both* is worth even more — while exact score magnitudes never enter the picture. Because it only ever compares positions, RRF is immune to the normalization headache. That robustness, plus a single gentle parameter, is why it's the merge built into many vector databases.

In [ ]:
def rrf(*rankings, k=60):
    """Reciprocal Rank Fusion. Each ranking is a list of doc indices, best
    first. A doc earns 1/(k+rank) from each list; we sum across lists. Merges
    by RANK, so no normalization and no alpha are needed."""
    fused = Counter()
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            fused[doc_id] += 1.0 / (k + rank)
    return fused


fused = rrf(dense_rank, sparse_rank)
rrf_rank = [i for i, _ in fused.most_common()]
rrf_scores = {i: fused[i] for i in range(len(CORPUS))}
show('HYBRID via RRF:', rrf_rank, rrf_scores)
print('E-4042 chunk (index 0) RRF rank:', rrf_rank.index(0) + 1, '-> rescued into the top 3.')

### The whole rescue, side by side

This is the headline demo from `rag_hybrid.py`, run end to end. Dense alone never surfaces the `E-4042` chunk in its top 3 (the code rounded off). Sparse ranks it first. **Either** fusion rescues it back into the top 3 — exactly the failure from the opening of this part, fixed, with maybe forty lines added to the app we already had.

In [ ]:
print(f'Query: {QUERY!r}')
print()

show('DENSE only (meaning):', dense_rank, dense)
show('SPARSE only (BM25 keywords):', sparse_rank, sparse)
show('HYBRID via RRF:', rrf_rank, rrf_scores)
show('HYBRID via weighted sum (alpha=0.5):', order(w), w)

print('The exact-code chunk is INVISIBLE in dense\'s top 3 (dense ranks it 5th).')
print('Sparse ranks it 1st. Either fusion rescues it into the top 3.')

And alpha really is a dial. Sweep it from pure dense to pure sparse and watch the `E-4042` chunk climb as we lean on keywords — the same effect the interactive figure in the essay lets you drag live.

In [ ]:
print('alpha  ->  rank of the E-4042 chunk (index 0) under weighted fusion')
for a in [1.0, 0.75, 0.5, 0.25, 0.0]:
    rank = order(weighted_fusion(dense, sparse, alpha=a)).index(0) + 1
    leans = 'pure dense' if a == 1.0 else 'pure sparse' if a == 0.0 else ''
    print(f'  alpha={a:<4}  rank {rank}   {leans}')

## Top-k is a real knob now

We've been treating **top-k** — the number of retrieved chunks we keep and pass to the generator — as a fixed constant. It isn't. It's a dial with an honest trade-off in both directions:

- Set **k too small** and you risk dropping the chunk that holds the answer (low recall, a starved generator). This was our opening failure: the answer at rank 5, cut off at 3.
- Set **k too large** and the prompt fills with marginally relevant chunks. That noise burns context budget, costs more, and — least intuitively — can make the answer *worse* via **lost in the middle**: models attend most to the beginning and end of their context and least to the middle, so a crucial chunk stranded mid-pile can be effectively ignored even though it was retrieved.

Let's make the trade-off concrete on our fused ranking. Suppose chunk 0 is the answer we must include.

In [ ]:
answer_idx = 0  # the E-4042 chunk literally answers the query
answer_rank = rrf_rank.index(answer_idx) + 1  # its position in the hybrid list

print(f'The answer (chunk {answer_idx}) sits at rank {answer_rank} in the hybrid list.')
print()
for k in [1, 2, 3, 6]:
    kept = rrf_rank[:k]
    has_answer = answer_idx in kept
    if not has_answer:
        note = 'answer DROPPED (low recall, generator starved)'
    elif k >= len(CORPUS):
        note = 'answer kept, but lots of noise (lost-in-the-middle risk)'
    else:
        note = 'answer kept, little noise (a good operating point)'
    print(f'  top-k={k}: kept ranks {kept}  ->  {note}')

print()
print('Resolution (Part 8): cast a wide net (large k), then RERANK down to the best few.')

## What you learned

- **Dense (semantic) retrieval** ranks by meaning — great at synonyms and paraphrase, but it blurs exact tokens (codes, IDs, names), rounding off the very keyword you needed.
- **Sparse (keyword) retrieval** scores by literal term overlap. **TF-IDF** weights terms by *frequency-here* times *rarity-everywhere*; **BM25** refines it with term-frequency saturation (`k1`) and document-length normalization (`b`), nailing exact matches but unable to bridge **vocabulary mismatch** (*car* vs. *automobile*).
- The two fail in **opposite directions** — which is exactly why combining them works.
- **Hybrid search** merges both. A **weighted sum** needs **score normalization** first (the scales differ) plus an **alpha** to trade them off; **Reciprocal Rank Fusion (RRF)** merges by rank, sidestepping normalization, which makes it a robust default.
- **Top-k** is a real knob: too small misses the answer; too large adds noise, wastes budget, and triggers **lost in the middle**.

A freshness note from the essay: BM25 and dense embeddings both have excellent off-the-shelf libraries (`rank_bm25`, `sentence-transformers`, and the hybrid mode built into most vector databases) whose APIs move fast. The hand-rolled BM25 here is for *understanding the mechanism* — before you ship, prefer a maintained library and verify its current usage. The fusion functions, though, are exactly what you'd use.

## Next

We cast a wide net in this part. **Part 8 — Making Retrieval Smarter** trims it expertly: **reranking with cross-encoders** (a slower, sharper model that re-scores the top candidates by reading query and chunk *together*), plus metadata filtering and query transformations — so the best chunk isn't just retrieved, it's ranked first.

← Previous: **Part 6 — Build Your First RAG**  ·  → Next: **Part 8 — Making Retrieval Smarter**